# Data Visualization with ggplot2

## What you'll learn
- Why visualization matters for data analysis
- The grammar of graphics: how ggplot2 thinks about charts
- How to create scatter plots, bar charts, line plots, histograms, and box plots
- How to customize colors, labels, themes, and facets
- How to save plots to files

## Prerequisites
- Completed Notebook 01 (R Fundamentals)

## Why Visualize Data?

Numbers alone can be misleading. A table of averages might look identical for two very different datasets. A chart reveals patterns, outliers, and relationships that numbers hide.

Visualization serves two purposes:
1. **Exploration** — finding patterns and anomalies in your data (for yourself)
2. **Communication** — presenting your findings to others

R's **ggplot2** package is one of the most powerful and popular visualization tools in any language.

## The Grammar of Graphics

ggplot2 is based on a system called the **grammar of graphics**. The idea: every chart is built from three components:

1. **Data** — the dataset you're plotting
2. **Aesthetics** (`aes`) — what you map to the x-axis, y-axis, color, size, etc.
3. **Geometries** (`geom_*`) — the visual shapes: points, bars, lines, etc.

You build a plot by combining these with `+`:
```r
ggplot(data, aes(x = ..., y = ...)) + geom_point()
```

Read this as: "Take this data, map these columns to x and y, and draw points."

In [ ]:
library(tidyverse)

# Load our datasets
employees <- read_csv("../data/employees.csv", show_col_types = FALSE)
sales <- read_csv("../data/sales.csv", show_col_types = FALSE)

## Scatter Plot: `geom_point()`

A scatter plot shows the relationship between two numeric variables. Each observation is a dot.

In [ ]:
# Age vs salary
ggplot(employees, aes(x = age, y = salary)) +
  geom_point()

In [ ]:
# Add color to show department
ggplot(employees, aes(x = age, y = salary, color = department)) +
  geom_point(size = 3)

## Bar Chart: `geom_col()` and `geom_bar()`

Bar charts show counts or totals across categories.

- `geom_bar()` — counts how many rows fall into each category (you only specify x)
- `geom_col()` — uses a value you provide for the bar height (you specify x and y)

In [ ]:
# Count employees per department (geom_bar counts for you)
ggplot(employees, aes(x = department)) +
  geom_bar()

In [ ]:
# Average salary by department (geom_col uses values you calculate)
dept_salary <- employees |>
  group_by(department) |>
  summarize(avg_salary = mean(salary))

ggplot(dept_salary, aes(x = department, y = avg_salary)) +
  geom_col()

In [ ]:
# Add color fill and reorder bars by value
ggplot(dept_salary, aes(x = reorder(department, avg_salary), y = avg_salary, fill = department)) +
  geom_col() +
  coord_flip()   # horizontal bars (easier to read long labels)

## Line Plot: `geom_line()`

Line plots show trends over time or a sequence. They connect data points with lines.

In [ ]:
# Monthly sales count over time
monthly_sales <- sales |>
  mutate(month = as.Date(paste0(substr(date, 1, 7), "-01"))) |>
  group_by(month) |>
  summarize(num_transactions = n())

ggplot(monthly_sales, aes(x = month, y = num_transactions)) +
  geom_line() +
  geom_point()   # add dots at each data point

In [ ]:
# Monthly revenue by category
monthly_by_cat <- sales |>
  mutate(
    month = as.Date(paste0(substr(date, 1, 7), "-01")),
    revenue = quantity * unit_price
  ) |>
  group_by(month, category) |>
  summarize(total_revenue = sum(revenue), .groups = "drop")

ggplot(monthly_by_cat, aes(x = month, y = total_revenue, color = category)) +
  geom_line(linewidth = 1)

## Histogram: `geom_histogram()`

A histogram shows the **distribution** of a single numeric variable — how often different value ranges occur.

In [ ]:
# Distribution of salaries
ggplot(employees, aes(x = salary)) +
  geom_histogram(bins = 15)

In [ ]:
# Distribution of unit prices in sales
ggplot(sales, aes(x = unit_price)) +
  geom_histogram(bins = 30, fill = "steelblue", color = "white")

## Box Plot: `geom_boxplot()`

A box plot compares the distribution of a numeric variable across categories. It shows the median, quartiles, and outliers.

In [ ]:
# Salary distribution by department
ggplot(employees, aes(x = department, y = salary, fill = department)) +
  geom_boxplot()

In [ ]:
# Unit price distribution by category
ggplot(sales, aes(x = category, y = unit_price, fill = category)) +
  geom_boxplot()

## Customizing Your Plots

### Labels and Titles

Use `labs()` to add a title, subtitle, and axis labels.

In [ ]:
ggplot(employees, aes(x = age, y = salary, color = department)) +
  geom_point(size = 3) +
  labs(
    title = "Employee Salary by Age",
    subtitle = "Colored by department",
    x = "Age (years)",
    y = "Annual Salary ($)",
    color = "Department"
  )

### Themes

Themes control the overall look of your plot. ggplot2 comes with several built-in themes.

In [ ]:
# Clean minimal theme
ggplot(employees, aes(x = age, y = salary)) +
  geom_point(size = 3, color = "steelblue") +
  labs(title = "theme_minimal()") +
  theme_minimal()

In [ ]:
# Classic theme (like base R plots)
ggplot(employees, aes(x = age, y = salary)) +
  geom_point(size = 3, color = "steelblue") +
  labs(title = "theme_classic()") +
  theme_classic()

In [ ]:
# Black and white theme (good for print)
ggplot(employees, aes(x = age, y = salary)) +
  geom_point(size = 3, color = "steelblue") +
  labs(title = "theme_bw()") +
  theme_bw()

### Colors

Colors can be mapped to data (`aes(color = ...)`) or set to a fixed value.

- **Inside `aes()`**: color varies by data (ggplot picks colors automatically)
- **Outside `aes()`**: one fixed color for all points

In [ ]:
# Fixed color (outside aes)
ggplot(employees, aes(x = age, y = salary)) +
  geom_point(color = "darkred", size = 3) +
  theme_minimal()

In [ ]:
# Color by data (inside aes) — note: 'color' for points/lines, 'fill' for bars/boxes
ggplot(employees, aes(x = department, fill = department)) +
  geom_bar() +
  theme_minimal()

### Facets — Small Multiples

**Facets** split your plot into multiple panels, one per category. This is great for comparing patterns across groups.

Use `facet_wrap(~variable)` to create a grid of panels.

In [ ]:
# Histogram of unit_price, one panel per category
ggplot(sales, aes(x = unit_price, fill = category)) +
  geom_histogram(bins = 20, color = "white") +
  facet_wrap(~category, scales = "free_x") +
  theme_minimal() +
  labs(title = "Price Distribution by Category")

In [ ]:
# Sales quantity by region, faceted by category
region_cat <- sales |>
  group_by(region, category) |>
  summarize(total_qty = sum(quantity), .groups = "drop")

ggplot(region_cat, aes(x = region, y = total_qty, fill = region)) +
  geom_col() +
  facet_wrap(~category) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Total Quantity Sold by Region and Category", y = "Total Quantity")

## Saving Plots

Use `ggsave()` to save a plot to a file. It automatically detects the format from the file extension (`.png`, `.pdf`, `.jpg`, etc.).

In [ ]:
# Create a plot and save it to a variable
salary_plot <- ggplot(employees, aes(x = age, y = salary, color = department)) +
  geom_point(size = 3) +
  labs(
    title = "Employee Salary by Age",
    x = "Age", y = "Salary ($)"
  ) +
  theme_minimal()

# Save it
ggsave("salary_by_age.png", salary_plot, width = 8, height = 5, dpi = 150)
cat("Plot saved to salary_by_age.png")

## Mini-Project: Tell a Story with the Sales Data

Create three visualizations that together tell a story about the sales data. Here's a guided approach:

**Chart 1:** Which category generates the most revenue?

In [ ]:
# Calculate revenue by category
cat_revenue <- sales |>
  mutate(revenue = quantity * unit_price) |>
  group_by(category) |>
  summarize(total_revenue = sum(revenue)) |>
  arrange(desc(total_revenue))

ggplot(cat_revenue, aes(x = reorder(category, total_revenue), y = total_revenue, fill = category)) +
  geom_col() +
  coord_flip() +
  labs(title = "Total Revenue by Category", x = "", y = "Revenue ($)") +
  theme_minimal() +
  theme(legend.position = "none")

**Chart 2:** How does revenue trend over time?

In [ ]:
monthly_revenue <- sales |>
  mutate(
    month = as.Date(paste0(substr(date, 1, 7), "-01")),
    revenue = quantity * unit_price
  ) |>
  group_by(month) |>
  summarize(total_revenue = sum(revenue))

ggplot(monthly_revenue, aes(x = month, y = total_revenue)) +
  geom_line(color = "steelblue", linewidth = 1) +
  geom_point(color = "steelblue", size = 2) +
  labs(title = "Monthly Revenue Over Time", x = "Month", y = "Revenue ($)") +
  theme_minimal()

**Chart 3:** How do regions compare across categories?

In [ ]:
region_revenue <- sales |>
  mutate(revenue = quantity * unit_price) |>
  group_by(region, category) |>
  summarize(total_revenue = sum(revenue), .groups = "drop")

ggplot(region_revenue, aes(x = region, y = total_revenue, fill = category)) +
  geom_col(position = "dodge") +
  labs(
    title = "Revenue by Region and Category",
    x = "Region", y = "Revenue ($)", fill = "Category"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

## Your Turn

**Exercise:** Create your own visualization using the employees or sales data. Try combining different geoms, themes, and facets. Some ideas:
- A box plot of salary by city
- A histogram of employee ages, faceted by department
- A scatter plot of quantity vs unit_price in the sales data

In [ ]:
# Your code here


## Clean Up

In [ ]:
# Remove the saved plot file
file.remove("salary_by_age.png")

---
## Summary

- ggplot2 uses the **grammar of graphics**: data + aesthetics + geometries
- Key geoms: `geom_point()`, `geom_col()`, `geom_line()`, `geom_histogram()`, `geom_boxplot()`
- Customize with `labs()`, themes (`theme_minimal()`, etc.), colors, and `facet_wrap()`
- Save plots with `ggsave()`

**Next up:** [07 - Creating Data Analysis Reports](07-reports.ipynb) — combine everything into a data analysis report.